# SHARP-LLM: CodeT5-Base Convergence Experiment
## Kaggle T4 Notebook — 5 Epochs, Seed 50

**Before running:** Add the `msbasanth/sharp-llm-processed-data` dataset via  
*Add Data → Your Datasets* so parquet files are available at `/kaggle/input/sharp-llm-processed-data/`.

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", result.stdout.strip() if result.returncode == 0 else "NOT FOUND")

# sentencepiece: required for CodeT5-Base RoBERTa tokenizer
# pyarrow/fastparquet: required to read parquet files
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentencepiece", "pyarrow", "fastparquet"], check=True)
print("✓ Dependencies ready")

In [ ]:
import os
from pathlib import Path

# Clone repo to working directory
REPO_DIR = Path("/kaggle/working/sharp-llm")

if not REPO_DIR.exists():
    result = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/msbasanth/sharp-llm.git",
         str(REPO_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        raise RuntimeError("Git clone failed")
    print("✓ Repo cloned")
else:
    result = subprocess.run(["git", "-C", str(REPO_DIR), "pull"], capture_output=True, text=True)
    print("✓ Repo updated:", result.stdout.strip())

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
import shutil

# Link processed data from Kaggle dataset input → repo data/processed/
DATA_SRC = Path("/kaggle/input/sharp-llm-processed-data")
DATA_DST = REPO_DIR / "data" / "processed"
DATA_DST.mkdir(parents=True, exist_ok=True)

FILES = ["train.parquet", "test.parquet", "label_map.json"]

for fname in FILES:
    src = DATA_SRC / fname
    dst = DATA_DST / fname
    if not dst.exists():
        if src.exists():
            shutil.copy(src, dst)
            print(f"✓ Copied {fname}")
        else:
            raise FileNotFoundError(
                f"{src} not found. Add 'msbasanth/sharp-llm-processed-data' as a dataset input."
            )
    else:
        print(f"✓ {fname} already present ({dst.stat().st_size // 1024} KB)")

In [ ]:
OUTPUT_DIR = "/kaggle/working/outputs/codet5-base/v1"

cmd = [
    sys.executable,
    "scripts/convergence_experiment_codet5_base.py",
    "--config", "config.yaml",
    "--model", "Salesforce/codet5-base",
    "--epochs", "5",
    "--seed", "50",
    "--batch-size", "8",
    "--learning-rate", "5e-5",
    "--output-dir", OUTPUT_DIR,
]

print("Command:", " ".join(cmd))
print("=" * 75)

result = subprocess.run(cmd, cwd=str(REPO_DIR))

if result.returncode == 0:
    print("\n✓ Experiment completed successfully!")
else:
    print(f"\n✗ Experiment failed (exit code {result.returncode})")

In [ ]:
import json

output_dir = Path(OUTPUT_DIR)
metrics_file = output_dir / "epoch_metrics.json"

if metrics_file.exists():
    with open(metrics_file) as f:
        metrics = json.load(f)

    print("=" * 75)
    print("CONVERGENCE RESULTS — CodeT5-Base (5 Epochs, Seed 50)")
    print("=" * 75)
    print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train F1':>8}  {'Val F1':>6}  {'Test F1':>7}  {'Test Acc':>8}  {'MCC':>6}")
    print("-" * 75)
    for m in metrics:
        print(f"{m['epoch']:>5}  {m['train_loss']:>10.4f}  {m['train_f1']:>8.4f}  "
              f"{m['val_f1']:>6.4f}  {m['test_f1']:>7.4f}  {m['test_accuracy']:>8.4f}  {m['test_mcc']:>6.4f}")

    final = metrics[-1]
    print(f"\nFinal Test F1:  {final['test_f1']:.4f}")
    print(f"Final Accuracy: {final['test_accuracy']:.4f}")
    print(f"Final MCC:      {final['test_mcc']:.4f}")

    # Print JSON for log recovery
    print("\nEPOCH_METRICS_JSON:", json.dumps(metrics))
else:
    print("metrics file not found — check training output above for errors")

In [ ]:
# List all output files available for download
print("Output files:")
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(output_dir)}  ({f.stat().st_size // 1024} KB)")